<a href="https://colab.research.google.com/github/ZzwahwahzZ/Conquer-Mobile-Assets./blob/main/notebooks/Conquer_Mobile_Color_Texture_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Conquer Mobile — اختبار الألوان والخامات (تشغيل كامل بخلية واحدة)

شغّل خلية التنفيذ الوحيدة بالأسفل. ستجهّز البيئة، تربط Drive، تجلب الصور والمجسمات المحفوظة، تطابق 3 عينات، تنشئ نسخًا ملوّنة وتقريرًا وZIP، وتحفظ نسخة على Drive ثم ترفع النتائج إلى GitHub وتتحقق منها.

**لا يعيد هذا الدفتر توليد المجسمات ولا يعدّل مجلدات الأصول الأصلية.** طريقة اللون الحالية اختبار أولي Vertex Colors بالإسقاط الأمامي، وليست خامة UV/PBR نهائية.

قبل التشغيل لأول مرة، أضف GitHub Personal Access Token في Colab Secrets بالاسم `GITHUB_TOKEN` مع صلاحية Contents: Read and write، وفعّل Notebook access. لا تضع المفتاح داخل الكود.


In [ ]:
#@title تشغيل مرحلة اختبار الألوان والخامات كاملة (خلية واحدة)
# لا يعيد توليد المجسمات ولا يغيّر ملفات الأصول الأصلية.
# يجهّز البيئة، يقرأ الصور وGLB من Drive/GitHub، يطابق العينات،
# يلوّن 3 مجسمات، ينشئ التقرير وZIP، ثم يرفع النتائج إلى GitHub.

import os, sys, re, json, csv, base64, zipfile, shutil, subprocess, unicodedata
from pathlib import Path

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install("trimesh>=4.4.0", "Pillow>=10", "numpy>=1.26", "requests>=2.31")
import numpy as np
import trimesh
import requests
from PIL import Image
from google.colab import drive, userdata
from IPython.display import display, Markdown

print("1/7 — تجهيز البيئة وربط Google Drive...")
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
REPO_NAME = "ZzwahwahzZ/Conquer-Mobile-Assets."
REPO_URL = "https://github.com/ZzwahwahzZ/Conquer-Mobile-Assets..git"
REPO_DIR = Path("/content/Conquer-Mobile-Assets")
WORK = Path("/content/Conquer_Color_Texture_Test")
RAW = WORK / "raw_models"
OUT = WORK / "colored_test"
for p in (WORK, RAW, OUT):
    p.mkdir(parents=True, exist_ok=True)

print("2/7 — جلب الصور الأصلية والعثور على أرشيف المجسمات...")
if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])

IMAGE_ROOT = REPO_DIR / "original_images"
if not IMAGE_ROOT.exists():
    raise FileNotFoundError(f"مجلد original_images غير موجود داخل المستودع: {IMAGE_ROOT}")
image_files = sorted(p for p in IMAGE_ROOT.rglob("*") if p.suffix.lower() in {".png",".jpg",".jpeg",".webp"})
if not image_files:
    raise RuntimeError("لم يتم العثور على صور أصلية.")

zip_candidates = list(DRIVE_ROOT.rglob("Conquer_TripoSR_42_Models.zip"))
if not zip_candidates:
    raise FileNotFoundError("لم أجد Conquer_TripoSR_42_Models.zip في Google Drive/MyDrive. ارفع الأرشيف المحفوظ إلى Drive ثم أعد تشغيل الخلية.")
MODEL_ZIP = zip_candidates[0]
with zipfile.ZipFile(MODEL_ZIP) as zf:
    glb_members = [n for n in zf.namelist() if n.lower().endswith(".glb")]
    if not glb_members:
        raise RuntimeError("أرشيف المجسمات لا يحتوي ملفات GLB.")
    for name in glb_members:
        # منع استخراج أي مسار خارج مجلد العمل
        target = (RAW / name).resolve()
        if RAW.resolve() not in target.parents and target != RAW.resolve():
            continue
        zf.extract(name, RAW)
glb_files = sorted(RAW.rglob("*.glb"))
if not glb_files:
    raise RuntimeError("لم يتم استخراج أي ملف GLB.")

print(f"تم العثور على {len(image_files)} صورة و{len(glb_files)} مجسم GLB.")
print("قائمة الصور:")
for i,p in enumerate(image_files): print(f"  [{i}] {p.relative_to(IMAGE_ROOT)}")
print("قائمة المجسمات:")
for i,p in enumerate(glb_files): print(f"  [{i}] {p.name}")

print("3/7 — مطابقة الصور بالمجسمات واختيار 3 عينات...")
def norm_name(path):
    s = unicodedata.normalize("NFKD", Path(path).stem).lower()
    s = re.sub(r"[^a-z0-9]+", "", s)
    for suffix in ("vertexcolor", "textured", "mesh", "model", "glb", "obj"):
        if s.endswith(suffix): s = s[:-len(suffix)]
    return s

mesh_by_key = {}
for m in glb_files:
    mesh_by_key.setdefault(norm_name(m), []).append(m)

pairs = []
used_meshes = set()
for img in image_files:
    key = norm_name(img)
    candidates = [m for m in mesh_by_key.get(key, []) if m not in used_meshes]
    if candidates:
        pairs.append((img, candidates[0]))
        used_meshes.add(candidates[0])

# إذا كانت الأسماء لا تتطابق حرفيًا لكن عدد الصور والمجسمات متساوٍ،
# نستخدم الترتيب الأبجدي المتناظر كما في دفعة TripoSR ذات الـ42 عنصرًا.
if len(pairs) < 3 and len(image_files) == len(glb_files):
    print("لم تكفِ مطابقة الأسماء؛ سأستخدم ترتيب القائمتين الأبجدي المتناظر للدفعة المتساوية.")
    pairs = list(zip(image_files, glb_files))

if len(pairs) < 3:
    raise RuntimeError(
        f"تعذر تحديد 3 أزواج مؤكدة تلقائيًا (المطابق: {len(pairs)}). "
        "راجع أسماء الصور وملفات GLB؛ لم يتم إنشاء أو تعديل أي أصل."
    )

SAMPLES = pairs[:3]
for i,(img,mesh) in enumerate(SAMPLES,1):
    print(f"العينة {i}: {img.relative_to(IMAGE_ROOT)}  <->  {mesh.name}")

print("4/7 — تطبيق ألوان الصور على المجسمات وتصدير GLB...")
def colorize_by_front_projection(mesh_path, image_path, out_path, image_size=1024):
    mesh = trimesh.load(mesh_path, force="mesh", process=False)
    if not isinstance(mesh, trimesh.Trimesh) or len(mesh.vertices) == 0 or len(mesh.faces) == 0:
        raise ValueError(f"هندسة غير صالحة: {mesh_path}")
    im = Image.open(image_path).convert("RGB")
    im.thumbnail((image_size, image_size))
    rgb = np.asarray(im)
    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    lo, hi = vertices.min(axis=0), vertices.max(axis=0)
    span = np.maximum(hi - lo, 1e-8)
    u = (vertices[:,0] - lo[0]) / span[0]
    v = (hi[1] - vertices[:,1]) / span[1]
    px = np.clip(np.rint(u * (rgb.shape[1]-1)).astype(int), 0, rgb.shape[1]-1)
    py = np.clip(np.rint(v * (rgb.shape[0]-1)).astype(int), 0, rgb.shape[0]-1)
    sampled = rgb[py, px]
    rgba = np.column_stack((sampled, np.full(len(sampled), 255, dtype=np.uint8)))
    mesh.visual = trimesh.visual.ColorVisuals(mesh=mesh, vertex_colors=rgba)
    mesh.export(out_path, file_type="glb")
    check = trimesh.load(out_path, force="mesh", process=False)
    if len(check.vertices) == 0 or len(check.faces) == 0:
        raise RuntimeError(f"فشل فحص الملف الناتج: {out_path}")
    return len(mesh.vertices), len(mesh.faces), Path(out_path).stat().st_size

report = []
for index,(image_path,mesh_path) in enumerate(SAMPLES,1):
    output_path = OUT / f"test_{index:02d}_{mesh_path.stem}_VERTEX_COLOR.glb"
    vertices,faces,size = colorize_by_front_projection(mesh_path,image_path,output_path)
    report.append({
        "sample": index,
        "source_image": str(image_path.relative_to(IMAGE_ROOT)),
        "source_glb": mesh_path.name,
        "output_glb": output_path.name,
        "vertices": vertices,
        "faces": faces,
        "bytes": size,
        "status": "SUCCESS"
    })
    print(f"[{index}/3] SUCCESS — {output_path.name} | vertices={vertices:,} | faces={faces:,} | {size:,} bytes")

report_path = OUT / "color_test_report.csv"
with report_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=list(report[0].keys()))
    writer.writeheader()
    writer.writerows(report)

metadata = {
    "project": "Conquer Mobile Assets",
    "stage": "3-model vertex-color proof of concept",
    "method": "front projection of source image colors onto mesh vertices",
    "note": "This is a preliminary vertex-color test, not final UV/PBR texturing.",
    "source_archive": MODEL_ZIP.name,
    "samples": report
}
metadata_path = OUT / "metadata.json"
metadata_path.write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")

print("5/7 — إنشاء أرشيف النتائج وحفظ نسخة في Google Drive...")
archive_base = "/content/Conquer_Color_Texture_Test_3_Samples"
archive_path = Path(shutil.make_archive(archive_base,"zip",root_dir=WORK,base_dir="colored_test"))
drive_copy = DRIVE_ROOT / archive_path.name
shutil.copy2(archive_path,drive_copy)
print("نسخة Drive:",drive_copy)

print("6/7 — تجهيز اتصال GitHub ورفع النتائج...")
try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None
if not github_token:
    raise RuntimeError(
        "الاختبار اكتمل وحُفظ على Drive، لكن لم أجد السر GITHUB_TOKEN في Colab Secrets. "
        "أضف GitHub Personal Access Token باسم GITHUB_TOKEN مع صلاحية Contents: Read and write، "
        "وفعّل Notebook access، ثم أعد تشغيل الخلية لرفع النتائج."
    )

api_root = f"https://api.github.com/repos/{REPO_NAME}/contents"
headers = {
    "Authorization": f"Bearer {github_token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}
upload_items = [(p, f"colored_test/{p.name}") for p in sorted(OUT.iterdir()) if p.is_file()]
upload_items.append((archive_path, "colored_test/Conquer_Color_Texture_Test_3_Samples.zip"))
uploaded = []
for local_path, repo_path in upload_items:
    content_b64 = base64.b64encode(local_path.read_bytes()).decode("ascii")
    check = requests.get(f"{api_root}/{repo_path}",headers=headers,timeout=60)
    if check.status_code == 200:
        sha = check.json()["sha"]
    elif check.status_code == 404:
        sha = None
    else:
        check.raise_for_status()
    payload = {
        "message": f"Add color texture test output: {Path(repo_path).name}",
        "content": content_b64,
        "branch": "main"
    }
    if sha: payload["sha"] = sha
    response = requests.put(f"{api_root}/{repo_path}",headers=headers,json=payload,timeout=180)
    response.raise_for_status()
    uploaded.append(repo_path)
    print("تم الرفع:",repo_path)

print("7/7 — التحقق من وجود الملفات المرفوعة على GitHub...")
verified = []
for repo_path in uploaded:
    response = requests.get(f"{api_root}/{repo_path}",headers=headers,timeout=60)
    response.raise_for_status()
    verified.append(repo_path)
    print("VERIFIED:",repo_path)

print("\nاكتمل اختبار الألوان والخامات بنجاح.")
print(f"المجسمات الناتجة: {sum(1 for p in OUT.glob('*.glb'))}")
print(f"ملفات تم رفعها والتحقق منها: {len(verified)}")
print("GitHub:",f"https://github.com/{REPO_NAME}/tree/main/colored_test")
print("Google Drive:",drive_copy)
print("\nتنبيه فني: الناتج الحالي Vertex Colors تجريبي وليس UV/PBR نهائيًا؛ نراجع العينات قبل تعميم الطريقة على الـ42 مجسمًا.")
